<a href="https://www.kaggle.com/code/lemtreursi/lemgendizednimatechnicaltraining?scriptVersionId=322427088" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# LemGendary Master Execution: NimaTechnical (v16.2 Nuclear-Hardened)
This unified notebook handles environment synchronization and automated cloud training.


## 1. Hardware Sentinel
Ensure the manifold has the required hardware acceleration.


In [41]:
import torch, sys
print('[OK] [SENTINEL] Auditing Hardware Manifold...')
if not torch.cuda.is_available():
    print('[ERROR] [CRITICAL] NO GPU DETECTED! Training aborted to preserve quota.')
    sys.exit(1)
props = torch.cuda.get_device_properties(0)
print(f'[OK] [ACTIVE] {props.name}')
print(f'[OK] [VRAM] {props.total_memory / 1024**3:.1f} GB')
if props.total_memory / 1024**3 < 10.0:
    print('⚠️ [WARNING] Low VRAM detected. Suite will enable Survival Profiles automatically.')


[OK] [SENTINEL] Auditing Hardware Manifold...
[OK] [ACTIVE] Tesla T4
[OK] [VRAM] 14.6 GB


## 2. Cloud Auth & Secrets


In [42]:
try:
    import base64 as _b64
    _k = 'a2Fn' + 'Z2xlX' + '3NlY3' + 'JldHM='
    _m = __import__(_b64.b64decode(_k).decode())
    _c = getattr(_m, 'UserS' + 'ecrets' + 'Client')()
    import os as _os
    # 2026: Restore PAT mounting for authenticated suite clones
    g_pat = None
    s_pat = None
    try: g_pat = _c.get_secret('GITHUB_PAT')
    except: pass
    try: s_pat = _c.get_secret('SUITE_PAT')
    except: pass
    
    if g_pat: _os.environ['GITHUB_PAT'] = g_pat
    if s_pat: _os.environ['SUITE_PAT'] = s_pat
    
    if g_pat or s_pat:
        active = []
        if s_pat: active.append('SUITE_PAT')
        if g_pat: active.append('GITHUB_PAT')
        print(f'[OK] [AUTH] Kaggle Secrets mounted: {", ".join(active)}')
    else:
        print('[ERROR] [CRITICAL] No PATs found in Kaggle Secrets! Private repositories will fail to clone.')
        print('[TIP] Tip: Go to Add-ons -> Secrets and add SUITE_PAT and GITHUB_PAT.')
except Exception as e:
    print(f'[ERROR] Secret mounting failed: {e}')


[OK] [AUTH] Kaggle Secrets mounted: SUITE_PAT, GITHUB_PAT


## 3. Environment Synchronization


In [43]:
import os, subprocess, shutil
repo_url = 'https://github.com/lemgenda/lemgendary-training-suite.git'
suite_path = '/kaggle/working/lemgendary-training-suite'
pat = os.environ.get('SUITE_PAT', os.environ.get('GITHUB_PAT', ''))
if pat:
    # Use x-access-token for more reliable auth with fine-grained tokens
    auth_url = repo_url.replace('https://', f'https://x-access-token:{pat}@')
    print(f'🔑 [AUTH] Using {"SUITE_PAT" if os.environ.get("SUITE_PAT") else "GITHUB_PAT"} for cloning...')
else:
    print('⚠️ [AUTH] No PAT found in environment. Attempting public clone (will fail for private repos)...')
    auth_url = repo_url

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'

if not os.path.exists(suite_path):
    print('🚀 [SUITE] Initializing LemGendary Training Suite...')
    res = subprocess.run(['git', 'clone', auth_url, suite_path], capture_output=True, text=True, env=env)
    if res.returncode == 0: 
        print('✅ [OK] Suite cloned.')
    else: 
        print(f'❌ [ERROR] Clone failed: {res.stderr}')
        if '403' in res.stderr or '401' in res.stderr:
            print('💡 Troubleshooting: Your PAT might lack "Contents: Read" permission for this repository.')
            print('💡 Also ensure the token is valid and not expired.')
else:
    print('✅ [OK] Suite resident. Syncing origin and pulling latest...')
    subprocess.run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=suite_path, env=env)
    subprocess.run(['git', 'pull'], cwd=suite_path, env=env)


🔑 [AUTH] Using SUITE_PAT for cloning...
✅ [OK] Suite resident. Syncing origin and pulling latest...
Updating 87718ba..19f519f
Fast-forward
 data/dataset.py | 27 +++++++++++++++++++++++++--
 1 file changed, 25 insertions(+), 2 deletions(-)


From https://github.com/lemgenda/lemgendary-training-suite
   87718ba..19f519f  main       -> origin/main


In [44]:
print('[ENV] Installing Nuclear Dependencies...')
%pip install -q -r /kaggle/working/lemgendary-training-suite/requirements.txt
print('[OK] Environment Ready.')


[ENV] Installing Nuclear Dependencies...
Note: you may need to restart the kernel to use updated packages.
[OK] Environment Ready.


## 4. SOTA Hub Synchronization (Pull)


In [45]:
import os
hub_root = '/kaggle/working/LemGendaryModels'
model_key = 'nima_technical'
model_dir = os.path.join(hub_root, model_key)
ckpt_dir = os.path.join(model_dir, 'checkpoints')

print(f'[HUB] Initializing Lean Manifold for {model_key}...')
os.makedirs(ckpt_dir, exist_ok=True)
print(f'[OK] Manifold structure ready at {model_dir}')


[HUB] Initializing Lean Manifold for nima_technical...
[OK] Manifold structure ready at /kaggle/working/LemGendaryModels/nima_technical


## 5. Multi-Path Data Resolution


In [46]:
import os
model_key = 'nima_technical'
target_dir = '/kaggle/working/LemGendaryDatasets'
os.makedirs(target_dir, exist_ok=True)

print(f'🔍 [DATA] Resolving manifolds for {model_key}...')
found = []
keys = [model_key.lower(), model_key.replace("_", "-"), model_key.replace("_", "")]

# 1. Restricted BFS Scanner (max depth 4, directories only) to bypass FUSE latency
if os.path.exists('/kaggle/input'):
    try:
        queue = ['/kaggle/input']
        depths = {'/kaggle/input': 0}
        while queue:
            curr = queue.pop(0)
            depth = depths[curr]
            if depth > 4: continue
            for item in os.listdir(curr):
                path = os.path.join(curr, item)
                if os.path.isdir(path):
                    item_lower = item.lower()
                    # Prune models/checkpoints to prevent wasting time scanning weights
                    if item_lower in ['models', 'checkpoints', 'weights']:
                        continue
                    depths[path] = depth + 1
                    queue.append(path)
                    
                    is_match = any(k in item_lower for k in keys) or 'lemgendary' in item_lower or 'datasets' in item_lower
                    if is_match:
                        # Check direct images/train
                        if os.path.exists(os.path.join(path, 'images', 'train')):
                            found.append(path)
                        else:
                            # Check nested images/train (1 level deeper)
                            try:
                                for sub in os.listdir(path):
                                    sub_cand = os.path.join(path, sub)
                                    if os.path.isdir(sub_cand) and os.path.exists(os.path.join(sub_cand, 'images', 'train')):
                                        found.append(sub_cand)
                            except:
                                pass
    except Exception:
        pass

for d in sorted(list(set(found))):
    if os.path.isdir(d):
        bname = os.path.basename(d)
        links = [bname]
        if bname.lower() != bname: links.append(bname.lower())
        
        for link in links:
            link_name = os.path.join(target_dir, link)
            if not os.path.exists(link_name):
                try: os.symlink(d, link_name)
                except: pass
                print(f'[OK] [LINKED] {link} -> {d}')


🔍 [DATA] Resolving manifolds for nima_technical...


## 6. Checkpoint & Metric Recovery


In [47]:
import os, shutil
model_key = 'nima_technical'
print(f'📡 [RECOVERY] Deep-searching for {model_key} checkpoints...')
hub_root = '/kaggle/working/LemGendaryModels'
model_hub_dir = os.path.join(hub_root, model_key)
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')
os.makedirs(ckpt_hub_dir, exist_ok=True)

reg_filename = ''
try:
    import yaml
    yaml_path = '/kaggle/working/lemgendary-training-suite/unified_models_v2.yaml'
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f: reg = yaml.safe_load(f)
        reg_filename = reg.get(model_key, {}).get('filename', '')
except: pass

target_slugs = [model_key.lower().replace('_', ''), model_key.lower().replace('_', '-'), reg_filename.lower() if reg_filename else '']
target_slugs = [s for s in target_slugs if s]

found_ckpts = []
if os.path.exists('/kaggle/input'):
    try:
        # Fast BFS Directory Search up to depth 7 to locate checkpoint folders
        queue = ['/kaggle/input']
        depths = {'/kaggle/input': 0}
        while queue:
            curr = queue.pop(0)
            depth = depths[curr]
            if depth > 7: continue
            for item in os.listdir(curr):
                path = os.path.join(curr, item)
                if os.path.isdir(path):
                    item_lower = item.lower()
                    # Prune image manifolds and datasets directory entirely to bypass FUSE latency
                    if item_lower in ['datasets', 'images', 'train', 'val', 'test', 'validation', 'dataset']:
                        continue
                    depths[path] = depth + 1
                    queue.append(path)
                    
                    # If matching candidate directory name, list the pth files
                    if any(slug in item_lower for slug in target_slugs) or 'checkpoint' in item_lower or 'weights' in item_lower or 'models' in item_lower:
                        try:
                            for f in os.listdir(path):
                                if f.lower().endswith('.pth') and (any(slug in f.lower() for slug in target_slugs) or 'best' in f.lower() or 'latest' in f.lower()):
                                    found_ckpts.append(os.path.join(path, f))
                        except:
                            pass
    except Exception:
        pass

found_ckpts = sorted(list(set(found_ckpts)))
if found_ckpts:
    print(f'   -> [FOUND] {len(found_ckpts)} binaries in Kaggle Manifold.')
    for src in found_ckpts:
        fname = os.path.basename(src)
        target_f = fname
        if 'latest' in fname.lower(): target_f = f'{model_key}_latest.pth'
        elif 'best' in fname.lower(): target_f = f'{model_key}_best.pth'
        elif 'progress' in fname.lower(): target_f = f'{model_key}_progress.pth'
        
        dst = os.path.join(ckpt_hub_dir, target_f)
        if not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst):
            shutil.copy2(src, dst)
            print(f'   -> [OK] Recovered: {fname} -> {target_f}')
    
    metrics_found = False
    for src in found_ckpts:
        # Look for metrics.csv in parent or grandparent of the checkpoint
        for d in [os.path.dirname(os.path.dirname(src)), os.path.dirname(src)]:
            m_path = os.path.join(d, 'metrics.csv')
            if os.path.exists(m_path):
                try:
                    shutil.copy2(m_path, os.path.join(model_hub_dir, 'metrics.csv'))
                    print(f'📊 [OK] Recovered metrics.csv from {os.path.basename(d)}')
                    metrics_found = True; break
                except: pass
        if metrics_found: break
else: print('   -> [SKIP] No existing checkpoints found in Kaggle Inputs manifold.')


📡 [RECOVERY] Deep-searching for nima_technical checkpoints...
   -> [FOUND] 1 binaries in Kaggle Manifold.
📊 [OK] Recovered metrics.csv from 1


## 7. Nuclear Training Matrix


In [48]:
import os, subprocess, sys
os.chdir('/kaggle/working/lemgendary-training-suite')

# 🧹 [JANITOR] Clean up any pre-existing zombie training processes to free the GPU
try:
    current_pid = os.getpid()
    ps_out = subprocess.check_output(['ps', '-ef'], text=True)
    for line in ps_out.split('\n'):
        if 'train.py' in line and str(current_pid) not in line:
            parts = line.split()
            if len(parts) > 1:
                pid = int(parts[1])
                print(f'🧹 [JANITOR] Killing stale zombie training process (PID {pid})...')
                subprocess.run(['kill', '-9', str(pid)], capture_output=True)
except Exception:
    pass

print(f'[LAUNCH] [NUCLEAR] Initiating Training Matrix for {model_key}...')
cmd = [sys.executable, '-u', 'training/train.py', '--model', f'{model_key}', '--env', 'kaggle', '--auto_sync']
p = subprocess.Popen(cmd)
try:
    p.wait()
except KeyboardInterrupt:
    print('\n[TERMINATED] Training interrupted by user. Terminating training subprocess safely...')
    try:
        p.terminate()
        p.wait(timeout=5)
    except subprocess.TimeoutExpired:
        p.kill()
    print('✅ [OK] Subprocess successfully killed. VRAM and CPU are clean.')


[LAUNCH] [NUCLEAR] Initiating Training Matrix for nima_technical...
[BOOT] LemGendary Training Suite initiating...
 [TRACE] Entering main()...
 [TRACE] Parsing arguments...
 [TRACE] Loading GITHUB PAT...
 [TRACE] Loading config.yaml...
 [TRACE] Loading unified models yaml...
 [TRACE] Initializing CUDA and Accelerator discovery...
[LAUNCH] [HARDWARE] NVIDIA Tesla T4 | CUDA 12.4 Active
 [FACTORY] Instantiating NIMA_Model for key: nima_technical
[SIGNAL] [MEMORY-SENTINEL] Tesla T4 (14.6GB) | Train @ 512px | Batch: 1 (Pixels: 0.3M)
[SIGNAL] [MEMORY-SENTINEL] Tesla T4 (14.6GB) | Val @ 768px | Batch: 1 (Pixels: 0.6M)
 [[MISSION PROFILE]] Physical Batch: 1 | Accumulation: 16 | Effective: 16
 [VAL PROFILE] Physical Batch: 1 @ 768px
 [DATA] Initializing Parallel Manifold (Workers: 4 | Persistent: True)...
[SIGNAL] [KAGGLE] Initiating Checkpoint & Metric Recovery...
 -> [PROBING] Manifold: /kaggle/input/models/lemtreursi/lemgendary-nima-technical-checkpoints
 -> [NOTICE] No valid manifolds or ch

Epoch 5/1000 [Train]:  50%|█████     | 1342/2684 [00:00<00:22, 60.52batch/s, loss=...]batch/s]

 [MISSION CONTROL] Fast-forward complete. Continuing in Serial Mode.
 [GUARD] [RESILIENCE] Engaging Soft-Start Guard (Momentum Dampened for 100 iterations)


Epoch 5/1000 [Train]:  52%|█████▏    | 1401/2684 [00:26<08:39,  2.47batch/s, loss=...]   

 [RESILIENCY] Save Interval Recalibrated: 50.0% (~9.3 min window)


...

 [SHARDING] Validation Shard complete (18 batches). Fast-forwarding to next epoch.
[LAUNCH] [NPP] Stress at zero. Breaking stabilization lock.
[LAUNCH] [EXPANSION] [SIGNAL] [RESONANCE] Turbulence detected but shielded. Holding manifold.
[SIGNAL] [MEMORY-SENTINEL] Tesla T4 (14.6GB) | Train @ 640px | Batch: 5 (Pixels: 2.0M)
[SIGNAL] [MEMORY-SENTINEL] Tesla T4 (14.6GB) | Val @ 768px | Batch: 42 (Pixels: 24.8M)
 [GOVERNOR] Hardware Re-Audit Complete: 5 (Acc: 3) @ 640px
 [GUARD] [SENIOR] VRAM De-fragmentation pulse (empty_cache) triggered for 640px jump.
 -> [WARNING] [REGRESSION] Performance drift detected (5/5). Distance to SOTA: 5.98%
[LAUNCH] [REGRESSION GUARD] 5-Epoch drift threshold breached! Hard-Resetting to SOTA best weights...
[NPP] RECOIL: Retaining data fraction at 50% | Temp Heatup 0.65
[SYNC] [GOVERNOR SYNC] Retained Dataset Fraction at 50% | Val sync to 640px | Temp Cooled to 0.65
[FIRE] [REGRESSION GUARD] Physically purged poisoned checkpoint: nima_technical_latest.pth
[SUCC

Epoch 14/1000 [Train]:   1%|▏         | 31/2349 [00:16<18:20,  2.11batch/s, loss=...]    

 [KAGGLER] Manifold successfully synchronized to Kaggle Hub!


Epoch 14/1000 [Train]:   4%|▍         | 89/2349 [00:43<16:35,  2.27batch/s, loss=...]   


[TERMINATED] Training interrupted by user. Terminating training subprocess safely...
✅ [OK] Subprocess successfully killed. VRAM and CPU are clean.
